In [15]:
#this code with reload any imports if you're updating them actively
%reload_ext autoreload
%autoreload 2
import pygem.pygem_input as pygem_prms
import scipy.optimize as opt
import numpy as np
import xarray as xr
import pandas as pd
import os
#PyGEM
import pygem.pygem_input as pygem_prms
import pygem.oggm_compat as oggm
import pygem.pygem_modelsetup as modelsetup
import class_climate

In [16]:
# read 10 year data files for each variable in varnames and merge
# 2010s surfrad and tcc are wrong files
# 1980s temp has wrong coordinates
eb_varnames = ['temp','dtemp','precip','surfrad','tcc','uwind','vwind']
i=3
varname = eb_varnames[i]
hourlyfp = '~/research/climate_data/ERA5/ERA5_hourly/varname/ERA5_varname_hourly'.replace('varname',varname)
da_0 = xr.open_dataarray(hourlyfp + '_80_89.nc')
da_1 = xr.open_dataarray(hourlyfp + '_90_99.nc')
if varname in ['temp']:
        da_1 = da_1[:(len(da_1)-24*366)]
da_2 = xr.open_dataarray(hourlyfp + '_00_09.nc')
da_3 = xr.open_dataarray(hourlyfp + '_10_21.nc')
print(da_0.coords)
print(da_1.coords)
print(da_2.coords)
print(da_3.coords)

#da = xr.merge([da_0,da_1,da_2,da_3])
#da.to_netcdf('~/research/climate_data/ERA5/ERA5_hourly/ERA5_varname_hourly.nc'.replace('varname',varname))
#print(da)
#368,164

Coordinates:
  * longitude  (longitude) float32 -145.5
  * latitude   (latitude) float32 63.34
  * time       (time) datetime64[ns] 1980-01-01 ... 1989-12-31T23:00:00
Coordinates:
  * longitude  (longitude) float32 -145.5
  * latitude   (latitude) float32 63.34
  * time       (time) datetime64[ns] 1990-01-01 ... 1999-12-31T23:00:00
Coordinates:
  * longitude  (longitude) float32 -145.5
  * latitude   (latitude) float32 63.34
  * time       (time) datetime64[ns] 2000-01-01 ... 2009-12-31T23:00:00
Coordinates:
  * longitude  (longitude) float32 -145.5
  * latitude   (latitude) float32 63.34
  * time       (time) datetime64[ns] 2010-01-01 ... 2021-12-31T23:00:00


In [23]:
gcm = class_climate.GCM(name='ERA5-hourly')
if pygem_prms.glac_no not in ['01.00570'] and not pygem_prms.run_eb:
    print('EB model can currently only run Gulkana glacier')
main_glac_rgi = modelsetup.selectglaciersrgitable(pygem_prms.glac_no,
                rgi_regionsO1=pygem_prms.rgi_regionsO1, rgi_regionsO2=pygem_prms.rgi_regionsO2,
                rgi_glac_number=pygem_prms.rgi_glac_number, include_landterm=pygem_prms.include_landterm,
                include_laketerm=pygem_prms.include_laketerm, include_tidewater=pygem_prms.include_tidewater)
    
# ===== TIME PERIOD =====
startyr = 1980
endyr = 2021
dates_table = modelsetup.datesmodelrun(startyear=startyr, endyear=endyr, spinupyears=pygem_prms.gcm_spinupyears,
            option_wateryear=pygem_prms.gcm_wateryear)

1 glaciers in region 1 are included in this model run: ['00570']
This study is focusing on 1 glaciers in region [1]
!! Not handling water years in modelsetup.datesmodelrun


In [24]:
%reload_ext autoreload
%autoreload 2
import class_climate

# ===== LOAD CLIMATE DATA =====
##gcm_prec, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.prec_fn, gcm.prec_vn, main_glac_rgi,dates_table)
#gcm_temp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.temp_fn, gcm.temp_vn, main_glac_rgi,dates_table)
##gcm_dtemp, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.dtemp_fn, gcm.dtemp_vn, main_glac_rgi,dates_table)

gcm_tcc, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.tcc_fn, gcm.tcc_vn, main_glac_rgi,dates_table)
gcm_surfrad, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.surfrad_fn, gcm.surfrad_vn, main_glac_rgi,dates_table) 
gcm_uwind, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.uwind_fn, gcm.uwind_vn, main_glac_rgi,dates_table)                                                      

# gcm_vwind, gcm_hours = gcm.importGCMvarnearestneighbor_xarray(gcm.vwind_fn, gcm.vwind_vn, main_glac_rgi,dates_table)

gcm_elev = gcm.importGCMfxnearestneighbor_xarray(gcm.elev_fn, gcm.elev_vn, main_glac_rgi)

u10
Check units of air temperature or precipitation


In [25]:
# READ GULKANA ELEVATION DATA FROM RGI 6.0
gulkana = '01.00570'
glac_no = [gulkana]

# ----- RGI DATA -----
# Filepath for RGI files
main_directory = os.getcwd()
glacier_table = modelsetup.selectglaciersrgitable(glac_no)
print('Gulkana Glacier Table:')
print(glacier_table)
elev_points = glacier_table[['Zmin','Zmed','Zmax']]

1 glaciers in region 1 are included in this model run: ['00570']
This study is focusing on 1 glaciers in region [1]
Gulkana Glacier Table:
   O1Index           RGIId   CenLon  CenLat  O1Region  O2Region    Area  Zmin  \
0      569  RGI60-01.00570 -145.427  63.281         1         2  17.567  1162   

   Zmax  Zmed  ...  Lmax  Connect  Form  TermType  Surging   RefDate   glacno  \
0  2438  1858  ...  8639        0     0         0        9  20090804  1.00570   

   rgino_str RGIId_float CenLon_360  
0   01.00570      1.0057    214.573  

[1 rows x 22 columns]


In [26]:
#%% MODEL PROPERTIES
density_ice = 900           # Density of ice [kg m-3] (or Gt / 1000 km3)
density_water = 1000        # Density of water [kg m-3]
area_ocean = 362.5 * 1e12   # Area of ocean [m2] (Cogley, 2012 from Marzeion et al. 2020)
k_ice = 2.33                # Thermal conductivity of ice [J s-1 K-1 m-1] recall (W = J s-1)
k_air = 0.023               # Thermal conductivity of air [J s-1 K-1 m-1] (Mellor, 1997)
k_air = 0.001               # Thermal conductivity of air [J s-1 K-1 m-1]
ch_ice = 1890000            # Volumetric heat capacity of ice [J K-1 m-3] (density=900, heat_capacity=2100 J K-1 kg-1)
ch_air = 1297               # Volumetric Heat capacity of air [J K-1 m-3] (density=1.29, heat_capacity=1005 J K-1 kg-1)
Lh_rf = 333550              # Latent heat of fusion [J kg-1]
tolerance = 1e-12           # Model tolerance (used to remove low values caused by rounding errors)
gravity = 9.81              # Gravity [m s-2]
pressure_std = 101325       # Standard pressure [Pa]
temp_std = 288.15           # Standard temperature [K]
R_gas = 8.3144598           # Universal gas constant [J mol-1 K-1]
molarmass_air = 0.0289644   # Molar mass of Earth's air [kg mol-1]


# MORE FACTORS THAT MIGHT BE NECESSARY
kp = 1                              # precipitation factor [-] (referred to as k_p in Radic etal 2013; c_prec in HH2015)
tbias = 5                           # temperature bias [deg C]
ddfsnow = 0.0041                    # degree-day factor of snow [m w.e. d-1 degC-1]
ddfsnow_iceratio = 0.7              # Ratio degree-day factor snow snow to ice
ddfice = ddfsnow / ddfsnow_iceratio # degree-day factor of ice [m w.e. d-1 degC-1]
precgrad = 0.0001                   # precipitation gradient on glacier [m-1]
lapserate = -0.0065                 # temperature lapse rate for both gcm to glacier and on glacier between elevation bins [K m-1]
tsnow_threshold = 1                 # temperature threshold for snow [deg C] (HH2015 used 1.5 degC +/- 1 degC)
calving_k = 0.7                     # frontal ablation rate [yr-1]

In [27]:
#READ GULKANA ELEVATIONS FROM OGGM GDIRS
#this step is specific to the EB for three points on Gulkana
#to generalize, need a variable geometry containing zwh for the points for the EB
gdir = oggm.single_flowline_glacier_directory(glac_no[0], logging_level='CRITICAL')
fls = oggm.get_glacier_zwh(gdir)
#filter out zero bins to get only initial glacier volume
fls = fls.iloc[np.nonzero(fls['h'].to_numpy())]
z_stats = np.array([np.min(fls['z']),np.median(fls['z']),np.max(fls['z'])])

print('Min, median and max bin elevations from OGGM flowlines:')
print(pd.Series(z_stats,index=['Zmin','Zmed','Zmax']))
print('Min, median and max elevation from RGI:')
print(elev_points)

#setup three points at minimum, median and maximum elevation band from OGGM
median_index = np.where(fls['z']==z_stats[1])[0][0]
w_stats = np.array([fls['w'][52],fls['w'][median_index],fls['w'][0]])
h_stats = np.array([fls['h'][52],fls['h'][median_index],fls['h'][0]])
geo_index = ['Bottom','Middle','Top']
geometry = pd.DataFrame({'z':z_stats,'w':w_stats,'h':h_stats},index=geo_index)

Min, median and max bin elevations from OGGM flowlines:
Zmin    1172.809436
Zmed    1628.401286
Zmax    2330.675374
dtype: float64
Min, median and max elevation from RGI:
   Zmin  Zmed  Zmax
0  1162  1858  2438


In [28]:
#manually set number of exponentially scaling bins
n_vert_bins = 10
n_points = len(geo_index)
option_bin = 0

#create variable to store glacier geometry
vert_bins = xr.Dataset(data_vars = dict(
    bin_depth = (['pt','vert_idx'],np.zeros((n_points,n_vert_bins))),
    bin_width = (['pt'],np.zeros(n_points)),
    bin_elev = (['pt'],np.zeros(n_points))),
    coords=dict(
        point=(['pt'],geo_index),
        vert_idx=range(n_vert_bins)
        )
    )

bin_depths = np.zeros((n_points,n_vert_bins))
#fill vertical bin heights based on ice thickness
for g in range(n_points):
    point = geo_index[g]
    #define surface elvation, width and ice thickness (m) of the current point
    pt_z, pt_w, pt_h = list(geometry.loc[point])
    vert_bins['bin_width'] = pt_w
    vert_bins['bin_elev'] = pt_z
    
    if option_bin==0:
        hs = [0.1,.25,.5,.75,1,2,5,10,20,pt_h-39.6]
        bin_depths[g,:] = hs
    else:
        c = opt.fsolve(lambda c: pt_h-np.sum(np.exp(np.arange(n_vert_bins)*c)),10)
        bin_depths[g,:] = np.exp(c*range(1,n_vert_bins))
vert_bins['bin_depth'] = (['pt','vert_idx'],bin_depths)

In [35]:
# lapserates - later have lapse rate adjust based on dew point temp
# currently let's just use a constant lapserate, but leave option:
# if pygem_prms.use_constant_lapserate:
gcm_lr = np.zeros(gcm_temp.shape) + pygem_prms.lapserate

#create dataset for variables that need to be adjusted
pt_climate = xr.Dataset(data_vars = dict(
    bin_T = (['pt','time'],np.zeros(n_points,len(gcm_hours))),
    bin_P = (['pt','time'],np.zeros(n_points,len(gcm_hours))),
    bin_snow = (['pt','time'],np.zeros(n_points,len(gcm_hours))),
    bin_elev = (['pt'],vert_bins['bin_elev'])),
    coords=dict(
        point=(['pt'],geo_index),
        vert_idx=range(n_vert_bins),
        time=gcm_hours
        ),
    attrs=dict(description="Climate data adjusted for points in EB.")
    )

pt_climate['bin_T'] = (['pt','time'],gcm_temp + gcm_lr * (gcm_elev - pt_climate['bin_elev']))
pt_climate['bin_P'] = (['pt','time'],gcm_prec)
pt_climate['bin_snow'] = (['pt','time'],gcm_temp.where(gcm_temp.t2m<2))

gcm_elev being handled stupidly for EB, revisit later


NameError: name 'gcm_temp' is not defined

In [46]:
Q_monthly = []

# ===== ENTER HOURLY LOOP!! =====
for hour in gcm_hours:
    if hour.is_month_start and hour.hour < 1:
        monthly_Q = 0
    Snet = gcm_surfrad[hour] * (1-albedo) #* (cos(theta))
    Qm = Snet + Lnet + Qs + Ql + Qp + Qg

1980-01-01 00:00:00
1980-02-01 00:00:00
1980-03-01 00:00:00
1980-04-01 00:00:00
1980-05-01 00:00:00
1980-06-01 00:00:00
1980-07-01 00:00:00
1980-08-01 00:00:00
1980-09-01 00:00:00
1980-10-01 00:00:00
1980-11-01 00:00:00
1980-12-01 00:00:00
1981-01-01 00:00:00
1981-02-01 00:00:00
1981-03-01 00:00:00
1981-04-01 00:00:00
1981-05-01 00:00:00
1981-06-01 00:00:00
1981-07-01 00:00:00
1981-08-01 00:00:00
1981-09-01 00:00:00
1981-10-01 00:00:00
1981-11-01 00:00:00
1981-12-01 00:00:00
1982-01-01 00:00:00
1982-02-01 00:00:00
1982-03-01 00:00:00
1982-04-01 00:00:00
1982-05-01 00:00:00
1982-06-01 00:00:00
1982-07-01 00:00:00
1982-08-01 00:00:00
1982-09-01 00:00:00
1982-10-01 00:00:00
1982-11-01 00:00:00
1982-12-01 00:00:00
1983-01-01 00:00:00
1983-02-01 00:00:00
1983-03-01 00:00:00
1983-04-01 00:00:00
1983-05-01 00:00:00
1983-06-01 00:00:00
1983-07-01 00:00:00
1983-08-01 00:00:00
1983-09-01 00:00:00
1983-10-01 00:00:00
1983-11-01 00:00:00
1983-12-01 00:00:00
1984-01-01 00:00:00
1984-02-01 00:00:00


In [ ]:
import pygem.pygem_input as pygem_prms
import pickle

In [ ]:
glacier_str = glac_no[0][1::]
modelprms_fn =  glacier_str + '-modelprms_dict.pkl'
modelprms_fp = pygem_prms.output_filepath + 'calibration/' + glacier_str.split('.')[0].zfill(2) + '/'
modelprms_fullfn = modelprms_fp + modelprms_fn
assert os.path.exists(modelprms_fullfn), glacier_str + ' calibrated parameters do not exist.'            
with open(modelprms_fullfn, 'rb') as f:
    modelprms_dict = pickle.load(f)